# BERD + BLIP: Report Supervision vs LLM-Reformatted VQA Supervision

This notebook runs two controlled fine-tuning experiments from the same checkpoint, `Salesforce/blip-vqa-base`.

**Experiment 1 — report supervision**

`image + canonical question -> original BERD caption`

**Experiment 2 — VQA supervision**

A small local LLM converts each BERD caption into short visual QA pairs, then:

`image + VQA question -> short answer`

The research question is whether VQA-style reformulation improves BLIP's bronchoscopy recognition/VQA accuracy.

## Fair comparison

Both models are evaluated on the **same untouched BERD test images** with:

1. **Primary benchmark:** BERD clinician-reviewed `label` field.
2. **Held-out generated VQA benchmark:** the same generated QA pairs for both models.
3. **Original caption benchmark:** BLEU, ROUGE-L, and METEOR against BERD captions.

The notebook defaults to the same number of optimizer updates for both experiments. This matters because VQA conversion creates several training rows per image.

Official sources:

- BERD dataset: https://doi.org/10.57760/sciencedb.28018
- BERD code: https://github.com/lxj22/BERD
- BERD paper: https://www.nature.com/articles/s41597-026-06692-8
- BLIP VQA: https://huggingface.co/Salesforce/blip-vqa-base

> Research use only; not for clinical decision-making.

## 1. Install dependencies

The notebook is intended for a CUDA notebook environment such as Colab or Kaggle.

In [2]:
%pip install -q "transformers>=4.56,<5" "accelerate>=1.0" "sentencepiece>=0.2" "scikit-learn>=1.4" "pandas>=2.0" "pillow>=10.0" "tqdm>=4.66" "matplotlib>=3.8" "nltk>=3.9" "rouge-score>=0.1.2" "sacrebleu>=2.4"

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and reproducibility

In [1]:
import os, re, gc, json, math, random, warnings
from pathlib import Path
from collections import Counter
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import precision_recall_fscore_support

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BlipProcessor,
    BlipForQuestionAnswering,
    get_cosine_schedule_with_warmup,
    set_seed,
)

import sacrebleu
import nltk
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score

warnings.filterwarnings("ignore")
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything()
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.7.1+cu118
CUDA: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


## 3. Configuration and BERD path

Download the official BERD `dataset.zip` and extract it to:

```text
dataset/
├── annotations/
│   ├── dataset_train.json
│   └── dataset_test.json
└── images/
    ├── 000001.png
    └── ...
```

By default the notebook expects `/content/dataset`. If `/content/dataset.zip` exists, it is extracted automatically.

In [2]:
DATA_ROOT = Path("C:\\Users\\Omen Max\\Datasets\\Bronchoscopic Datasets\\BERD\\dataset")
OPTIONAL_ZIP = Path("C:\\Users\\Omen Max\\Datasets\\Bronchoscopic Datasets\\BERD\\dataset.zip")

if OPTIONAL_ZIP.exists() and not DATA_ROOT.exists():
    import zipfile
    with zipfile.ZipFile(OPTIONAL_ZIP, "r") as zf:
        zf.extractall("C:\\Users\\Omen Max\\Datasets\\Bronchoscopic Datasets\\BERD\\dataset")
    print("Extracted:", OPTIONAL_ZIP)

TRAIN_JSON = DATA_ROOT / "annotations" / "dataset_train.json"
TEST_JSON = DATA_ROOT / "annotations" / "dataset_test.json"
IMAGE_DIR = DATA_ROOT / "images"

BLIP_MODEL_NAME = "Salesforce/blip-vqa-base"
SMALL_LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

CANONICAL_QUESTION = "What abnormalities are visible in this bronchoscopy image?"

REPORT_EPOCHS = 5
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MAX_QUESTION_TOKENS = 48
MAX_REPORT_TOKENS = 128
MAX_VQA_ANSWER_TOKENS = 32

EQUAL_OPTIMIZER_BUDGET = True
FREEZE_VISION_ENCODER = False
USE_GRADIENT_CHECKPOINTING = True

LLM_BATCH_SIZE = 32
LLM_MAX_NEW_TOKENS = 220
N_ADDITIONAL_QA = 2

VQA_CACHE_DIR = Path("/content/berd_vqa_cache")
VQA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DEBUG_MAX_TRAIN_ROWS = None
DEBUG_MAX_TEST_ROWS = None

OUTPUT_ROOT = Path("/content/berd_blip_experiments")
EXP1_DIR = OUTPUT_ROOT / "exp1_report_supervision"
EXP2_DIR = OUTPUT_ROOT / "exp2_vqa_supervision"
RESULTS_DIR = OUTPUT_ROOT / "results"
for p in [EXP1_DIR, EXP2_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

DATA_ROOT: C:\Users\Omen Max\Datasets\Bronchoscopic Datasets\BERD\dataset
OUTPUT_ROOT: \content\berd_blip_experiments


## 4. Load and validate BERD

In [3]:
REQUIRED_KEYS = {
    "image_path", "image_id", "caption", "location",
    "width", "height", "label", "patient_id"
}

def read_json_list(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Download/extract BERD and update DATA_ROOT."
        )
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON list in {path}")
    return data

def resolve_image_path(item):
    raw = Path(str(item["image_path"]))
    candidates = [
        DATA_ROOT / raw,
        IMAGE_DIR / raw.name,
        DATA_ROOT / "images" / raw.name,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return str(IMAGE_DIR / raw.name)

def normalize_label_string(x):
    parts = x if isinstance(x, list) else str(x).split(",")
    parts = [re.sub(r"\s+", " ", str(p).strip().lower()) for p in parts if str(p).strip()]
    return ", ".join(parts)

def to_dataframe(items):
    rows = []
    for item in items:
        missing = REQUIRED_KEYS - set(item)
        if missing:
            raise ValueError(f"Missing fields {missing} in item {item}")
        row = dict(item)
        row["image_file"] = resolve_image_path(item)
        row["caption"] = str(row["caption"]).strip()
        row["label_norm"] = normalize_label_string(row["label"])
        row["patient_id"] = str(row["patient_id"])
        rows.append(row)
    return pd.DataFrame(rows)

train_all_df = to_dataframe(read_json_list(TRAIN_JSON))
test_df = to_dataframe(read_json_list(TEST_JSON))

if DEBUG_MAX_TRAIN_ROWS:
    train_all_df = train_all_df.iloc[:DEBUG_MAX_TRAIN_ROWS].copy()
if DEBUG_MAX_TEST_ROWS:
    test_df = test_df.iloc[:DEBUG_MAX_TEST_ROWS].copy()

print("Official train images:", len(train_all_df))
print("Official test images :", len(test_df))
print("Train patients:", train_all_df.patient_id.nunique())
print("Test patients :", test_df.patient_id.nunique())

overlap = set(train_all_df.patient_id) & set(test_df.patient_id)
print("Train/test patient overlap:", len(overlap))
assert not overlap, "Patient leakage detected between official train and test."

missing_images = [
    p for p in pd.concat([train_all_df.image_file, test_df.image_file])
    if not Path(p).exists()
]
print("Missing images:", len(missing_images))
if missing_images:
    print(missing_images[:5])
    raise FileNotFoundError("Some image paths could not be resolved.")

Official train images: 6014
Official test images : 316
Train patients: 3512
Test patients : 180
Train/test patient overlap: 0
Missing images: 0


### Create a patient-disjoint validation subset

The official test set remains untouched. Validation is carved only from the official training set and grouped by `patient_id`.

In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
train_idx, val_idx = next(gss.split(train_all_df, groups=train_all_df["patient_id"]))

train_df = train_all_df.iloc[train_idx].reset_index(drop=True)
val_df = train_all_df.iloc[val_idx].reset_index(drop=True)

assert not (set(train_df.patient_id) & set(val_df.patient_id))
assert not (set(train_df.patient_id) & set(test_df.patient_id))
assert not (set(val_df.patient_id) & set(test_df.patient_id))

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Patients:", train_df.patient_id.nunique(), val_df.patient_id.nunique(), test_df.patient_id.nunique())

Train: 5400 Val: 614 Test: 316
Patients: 3160 352 180


### Inspect labels

In [5]:
def split_labels(label_norm):
    return [x.strip() for x in str(label_norm).split(",") if x.strip()]

label_counts = Counter()
for x in train_all_df["label_norm"]:
    label_counts.update(split_labels(x))

display(pd.DataFrame(label_counts.most_common(), columns=["label", "count"]))
display(train_all_df[["image_id", "caption", "label_norm", "patient_id"]].head(10))

,label,count
0,congested,2238
1,edematous,1674
2,sputum,1136
3,normal,1070
4,narrow,866
5,neoplasm,768
6,blood,451
7,external pressure,298
8,rough,254
9,nodules,252


,image_id,caption,label_norm,patient_id
0,000001,It can be seen that the old dark red blood clo...,"clot, congested, edematous",002631
1,000002,A large amount of white sticky phlegm,sputum,001559
2,000003,It is normal,normal,001739
3,000005,"The opening is stenotic, the surrounding mucos...","narrow, rough, congested, edematous",002728
4,000006,Stenosis of the opening and edema of the surro...,"narrow, edematous",000748
5,000007,It is normal,normal,003496
6,000008,It is normal,normal,001551
7,000009,It is normal,normal,003304
8,000011,"Mucosal congestion, edema","congested, edematous",000028
9,000012,"Mucosa is smooth, congested and swollen","congested, edematous",002535


## 5. Experiment 1 records — original report/caption supervision

Every image uses the same question and the original BERD caption as its answer.

In [6]:
def build_report_records(df):
    return [
        {
            "image_id": str(r.image_id),
            "patient_id": str(r.patient_id),
            "image_file": str(r.image_file),
            "question": CANONICAL_QUESTION,
            "answer": str(r.caption),
            "gold_labels": str(r.label_norm),
            "source_caption": str(r.caption),
            "task": "report",
        }
        for r in df.itertuples(index=False)
    ]

report_train_records = build_report_records(train_df)
report_val_records = build_report_records(val_df)
report_test_records = build_report_records(test_df)

print("Experiment 1 train rows:", len(report_train_records))
display(pd.DataFrame(report_train_records).head())

Experiment 1 train rows: 5400


,image_id,patient_id,image_file,question,answer,gold_labels,source_caption,task
0,000001,002631,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,It can be seen that the old dark red blood clo...,"clot, congested, edematous",It can be seen that the old dark red blood clo...,report
1,000002,001559,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,A large amount of white sticky phlegm,sputum,A large amount of white sticky phlegm,report
2,000003,001739,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,It is normal,normal,It is normal,report
3,000005,002728,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,"The opening is stenotic, the surrounding mucos...","narrow, rough, congested, edematous","The opening is stenotic, the surrounding mucos...",report
4,000006,000748,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,Stenosis of the opening and edema of the surro...,"narrow, edematous",Stenosis of the opening and edema of the surro...,report


## 6. Use a small local LLM to create VQA supervision

The small LLM only decomposes the caption into additional short visual QA pairs. The canonical abnormality question uses BERD's existing clinician-reviewed label as the answer, which gives the experiment a stable gold target and avoids making an LLM-generated answer the only ground truth.

In [7]:
OFFICIAL_BERD_LABELS = [
    "neoplasm", "narrow", "blood", "clot", "sputum", "rough", "normal",
    "infiltration changes", "congested", "edematous", "widened",
    "external pressure", "surgical stump", "nodules", "fistula",
    "postoperative change", "granulation", "necrotic", "mass",
    "pigmentation", "tube", "ulcer"
]

SYSTEM_PROMPT = "\n".join([
    "You convert a single-image bronchoscopy caption into concise visual question-answer supervision.",
    "",
    "Rules:",
    "1. Use ONLY information explicitly present in the caption and supplied clinician-reviewed labels.",
    "2. Do not infer diagnosis, pathology, history, treatment, prognosis, or measurements.",
    "3. Questions must be answerable from the image according to the supplied annotation.",
    "4. Answers must be concise, ideally 1-10 words.",
    "5. Do not ask about metadata or patient identity.",
    "6. Do not add anatomical location unless the caption itself explicitly describes it.",
    f"7. Output valid JSON only: a JSON array of exactly {N_ADDITIONAL_QA} objects.",
    '8. Each object must contain exactly two string fields: "question" and "answer".',
    f'9. Do NOT repeat this canonical question: "{CANONICAL_QUESTION}".',
    "10. Prefer questions about visible morphology, mucosa, obstruction, bleeding, secretions, masses, nodules, edema, congestion, narrowing, or normality.",
    "",
    "Allowed BERD labels: " + ", ".join(OFFICIAL_BERD_LABELS),
])

def make_llm_user_prompt(caption, gold_labels):
    return (
        "Caption:\n" + caption +
        "\n\nClinician-reviewed labels:\n" + gold_labels +
        f"\n\nCreate exactly {N_ADDITIONAL_QA} additional visual QA pairs."
    )

print(SYSTEM_PROMPT)

You convert a single-image bronchoscopy caption into concise visual question-answer supervision.

Rules:
1. Use ONLY information explicitly present in the caption and supplied clinician-reviewed labels.
2. Do not infer diagnosis, pathology, history, treatment, prognosis, or measurements.
3. Questions must be answerable from the image according to the supplied annotation.
4. Answers must be concise, ideally 1-10 words.
5. Do not ask about metadata or patient identity.
6. Do not add anatomical location unless the caption itself explicitly describes it.
7. Output valid JSON only: a JSON array of exactly 2 objects.
8. Each object must contain exactly two string fields: "question" and "answer".
9. Do NOT repeat this canonical question: "What abnormalities are visible in this bronchoscopy image?".
10. Prefer questions about visible morphology, mucosa, obstruction, bleeding, secretions, masses, nodules, edema, congestion, narrowing, or normality.

Allowed BERD labels: neoplasm, narrow, blood,

### Load the small LLM

In [8]:
def load_small_llm():
    tok = AutoTokenizer.from_pretrained(SMALL_LLM_NAME)
    tok.padding_side = "left"  # <-- Required for decoder-only batched generation
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

    # Force entire model explicitly to GPU instead of device_map="auto"
    mdl = AutoModelForCausalLM.from_pretrained(
        SMALL_LLM_NAME,
        torch_dtype=dtype
    ).to(device)
    
    mdl.eval()
    return tok, mdl

llm_tokenizer, llm_model = load_small_llm()
print("Loaded", SMALL_LLM_NAME)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded Qwen/Qwen2.5-0.5B-Instruct


### Parse and validate LLM JSON

In [9]:
def extract_json_array(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    start, end = text.find("["), text.rfind("]")
    if start < 0 or end <= start:
        return None
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return None

def sanitize_qa_list(raw, expected_n=N_ADDITIONAL_QA):
    if not isinstance(raw, list):
        return []
    clean, seen = [], set()
    for obj in raw:
        if not isinstance(obj, dict):
            continue
        q = re.sub(r"\s+", " ", str(obj.get("question", "")).strip())
        a = re.sub(r"\s+", " ", str(obj.get("answer", "")).strip())
        if not q or not a:
            continue
        if q.lower() == CANONICAL_QUESTION.lower():
            continue
        if len(a.split()) > 14:
            continue
        key = (q.lower(), a.lower())
        if key in seen:
            continue
        seen.add(key)
        clean.append({"question": q, "answer": a})
        if len(clean) == expected_n:
            break
    return clean

def canonical_gold_qa(row):
    return {
        "question": CANONICAL_QUESTION,
        "answer": row["label_norm"],
    }

### Batched deterministic VQA conversion

In [11]:
@torch.inference_mode()
def llm_generate_batch(captions, labels, max_new_tokens=120):
    conversations = [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_llm_user_prompt(c, l)},
        ]
        for c, l in zip(captions, labels)
    ]

    texts = [
        llm_tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
        for conv in conversations
    ]

    batch = llm_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,  # Captions and prompt are short; 512 is plenty
    )

    device = next(llm_model.parameters()).device
    batch = {k: v.to(device) for k, v in batch.items()}

    generated = llm_model.generate(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=llm_tokenizer.pad_token_id,
        eos_token_id=llm_tokenizer.eos_token_id,
    )

    input_len = batch["input_ids"].shape[1]
    return llm_tokenizer.batch_decode(generated[:, input_len:], skip_special_tokens=True)

def convert_split_to_vqa(df, split_name, batch_size=LLM_BATCH_SIZE, force_rebuild=False):
    cache_path = VQA_CACHE_DIR / f"{split_name}_vqa.jsonl"

    if cache_path.exists() and not force_rebuild:
        print("Loading cached", cache_path)
        with open(cache_path, "r", encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]

    rows = df.to_dict("records")
    all_records = []

    with open(cache_path, "w", encoding="utf-8") as fout:
        for start in tqdm(range(0, len(rows), batch_size), desc=f"LLM -> {split_name}"):
            batch_rows = rows[start:start + batch_size]
            outputs = llm_generate_batch(
                [r["caption"] for r in batch_rows],
                [r["label_norm"] for r in batch_rows],
            )

            for row, raw_text in zip(batch_rows, outputs):
                qa_pairs = [canonical_gold_qa(row)]
                qa_pairs += sanitize_qa_list(extract_json_array(raw_text))

                for qa_idx, qa in enumerate(qa_pairs):
                    rec = {
                        "image_id": str(row["image_id"]),
                        "patient_id": str(row["patient_id"]),
                        "image_file": str(row["image_file"]),
                        "question": qa["question"],
                        "answer": qa["answer"],
                        "gold_labels": str(row["label_norm"]),
                        "source_caption": str(row["caption"]),
                        "qa_index": qa_idx,
                        "is_canonical": qa_idx == 0,
                        "task": "vqa",
                        "llm_model": SMALL_LLM_NAME if qa_idx > 0 else None,
                    }
                    all_records.append(rec)
                    fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fout.flush()

    print("Saved", cache_path)
    return all_records

vqa_train_records = convert_split_to_vqa(train_df, "train")
vqa_val_records = convert_split_to_vqa(val_df, "val")
vqa_test_records = convert_split_to_vqa(test_df, "test")

print("VQA rows:", len(vqa_train_records), len(vqa_val_records), len(vqa_test_records))
display(pd.DataFrame(vqa_train_records).head(12))

Loading cached \content\berd_vqa_cache\train_vqa.jsonl
Loading cached \content\berd_vqa_cache\val_vqa.jsonl
Loading cached \content\berd_vqa_cache\test_vqa.jsonl
VQA rows: 8 0 0


,image_id,patient_id,image_file,question,answer,gold_labels,source_caption,qa_index,is_canonical,task,llm_model
0,000001,002631,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,"clot, congested, edematous","clot, congested, edematous",It can be seen that the old dark red blood clo...,0,True,vqa,None
1,000002,001559,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,sputum,sputum,A large amount of white sticky phlegm,0,True,vqa,None
2,000003,001739,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,normal,normal,It is normal,0,True,vqa,None
3,000005,002728,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,"narrow, rough, congested, edematous","narrow, rough, congested, edematous","The opening is stenotic, the surrounding mucos...",0,True,vqa,None
4,000006,000748,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,"narrow, edematous","narrow, edematous",Stenosis of the opening and edema of the surro...,0,True,vqa,None
5,000007,003496,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,normal,normal,It is normal,0,True,vqa,None
6,000008,001551,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,normal,normal,It is normal,0,True,vqa,None
7,000009,003304,C:\Users\Omen Max\Datasets\Bronchoscopic Datas...,What abnormalities are visible in this broncho...,normal,normal,It is normal,0,True,vqa,None


### Audit generated VQA pairs

Review these before training. If they contain non-visual inference or incorrect medical facts, revise the prompt and rebuild the cache with `force_rebuild=True`.

In [12]:
audit_df = pd.DataFrame(vqa_train_records)
extra = audit_df[~audit_df["is_canonical"]]
if len(extra):
    display(
        extra.sample(min(30, len(extra)), random_state=SEED)[
            ["image_id", "source_caption", "question", "answer", "gold_labels"]
        ]
    )
else:
    print("No valid additional QA pairs were generated.")

No valid additional QA pairs were generated.


### Free the small LLM before loading BLIP

In [13]:
del llm_model, llm_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Small LLM removed from memory.")

Small LLM removed from memory.


## 7. BLIP dataset and collators

In [14]:
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_NAME)

class BerdBlipDataset(Dataset):
    def __init__(self, records):
        self.records = records
    def __len__(self):
        return len(self.records)
    def __getitem__(self, idx):
        return self.records[idx]

class BlipTrainCollator:
    def __init__(self, processor, max_answer_tokens):
        self.processor = processor
        self.max_answer_tokens = max_answer_tokens

    def __call__(self, batch):
        images = [Image.open(x["image_file"]).convert("RGB") for x in batch]
        questions = [x["question"] for x in batch]
        answers = [x["answer"] for x in batch]

        model_inputs = self.processor(
            images=images,
            text=questions,
            padding=True,
            truncation=True,
            max_length=MAX_QUESTION_TOKENS,
            return_tensors="pt",
        )

        target = self.processor.tokenizer(
            answers,
            padding=True,
            truncation=True,
            max_length=self.max_answer_tokens,
            return_tensors="pt",
        )
        labels = target["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        model_inputs["labels"] = labels
        return model_inputs

class BlipInferenceCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        images = [Image.open(x["image_file"]).convert("RGB") for x in batch]
        questions = [x["question"] for x in batch]
        model_inputs = self.processor(
            images=images,
            text=questions,
            padding=True,
            truncation=True,
            max_length=MAX_QUESTION_TOKENS,
            return_tensors="pt",
        )
        return {"model_inputs": model_inputs, "records": batch}

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

## 8. Training utilities

In [15]:
def device_and_amp():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        amp_dtype = torch.float32
    return device, amp_dtype

def build_blip_model():
    seed_everything(SEED)
    model = BlipForQuestionAnswering.from_pretrained(BLIP_MODEL_NAME)

    if FREEZE_VISION_ENCODER:
        for p in model.vision_model.parameters():
            p.requires_grad = False

    if USE_GRADIENT_CHECKPOINTING:
        try:
            model.gradient_checkpointing_enable()
            print("Gradient checkpointing enabled.")
        except Exception as e:
            print("Gradient checkpointing unavailable:", e)

    return model

@torch.inference_mode()
def evaluate_loss(model, loader, device, amp_dtype):
    model.eval()
    losses = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype if device.type == "cuda" else torch.float32,
            enabled=(device.type == "cuda"),
        ):
            outputs = model(**batch)
        losses.append(float(outputs.loss.detach().cpu()))
    return float(np.mean(losses)) if losses else float("nan")

def train_blip(train_records, val_records, output_dir, max_answer_tokens, max_optimizer_steps):
    seed_everything(SEED)
    device, amp_dtype = device_and_amp()

    train_ds = BerdBlipDataset(train_records)
    val_ds = BerdBlipDataset(val_records)
    collator = BlipTrainCollator(blip_processor, max_answer_tokens)

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(
        train_ds,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=2,
        pin_memory=True,
        collate_fn=collator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collator,
    )

    model = build_blip_model().to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(max_optimizer_steps * WARMUP_RATIO),
        num_training_steps=max_optimizer_steps,
    )

    use_scaler = device.type == "cuda" and amp_dtype == torch.float16
    scaler = torch.cuda.amp.GradScaler(enabled=use_scaler)

    best_val = float("inf")
    global_step = 0
    micro_step = 0
    epoch = 0
    history = []
    optimizer.zero_grad(set_to_none=True)

    while global_step < max_optimizer_steps:
        epoch += 1
        model.train()
        running = []

        pbar = tqdm(train_loader, desc=f"epoch {epoch}")
        for batch in pbar:
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

            with torch.autocast(
                device_type=device.type,
                dtype=amp_dtype if device.type == "cuda" else torch.float32,
                enabled=(device.type == "cuda"),
            ):
                outputs = model(**batch)
                loss = outputs.loss / GRAD_ACCUM_STEPS

            if use_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running.append(float(loss.detach().cpu()) * GRAD_ACCUM_STEPS)
            micro_step += 1

            if micro_step % GRAD_ACCUM_STEPS == 0:
                if use_scaler:
                    scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)

                if use_scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1

                pbar.set_postfix(
                    loss=np.mean(running[-20:]),
                    optimizer_step=global_step,
                    max_steps=max_optimizer_steps,
                )

                if global_step >= max_optimizer_steps:
                    break

        val_loss = evaluate_loss(model, val_loader, device, amp_dtype)
        row = {
            "epoch": epoch,
            "optimizer_step": global_step,
            "train_loss": float(np.mean(running)) if running else np.nan,
            "val_loss": val_loss,
        }
        history.append(row)
        print(row)

        if val_loss < best_val:
            best_val = val_loss
            output_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(output_dir, safe_serialization=True)
            blip_processor.save_pretrained(output_dir)
            pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)
            print("Saved best checkpoint ->", output_dir)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    best_model = BlipForQuestionAnswering.from_pretrained(output_dir).to(device)
    best_model.eval()
    return best_model, pd.DataFrame(history)

### Equal optimizer-step budget

In [16]:
micro_batches_per_epoch = math.ceil(len(report_train_records) / TRAIN_BATCH_SIZE)
optimizer_steps_per_epoch = math.ceil(micro_batches_per_epoch / GRAD_ACCUM_STEPS)
BASELINE_MAX_STEPS = optimizer_steps_per_epoch * REPORT_EPOCHS

if EQUAL_OPTIMIZER_BUDGET:
    EXP1_MAX_STEPS = BASELINE_MAX_STEPS
    EXP2_MAX_STEPS = BASELINE_MAX_STEPS
else:
    EXP1_MAX_STEPS = BASELINE_MAX_STEPS
    vqa_micro_batches = math.ceil(len(vqa_train_records) / TRAIN_BATCH_SIZE)
    EXP2_MAX_STEPS = math.ceil(vqa_micro_batches / GRAD_ACCUM_STEPS) * REPORT_EPOCHS

print("Exp1 optimizer steps:", EXP1_MAX_STEPS)
print("Exp2 optimizer steps:", EXP2_MAX_STEPS)

Exp1 optimizer steps: 1690
Exp2 optimizer steps: 1690


## 9. Experiment 1 — train on original BERD captions

In [ ]:
exp1_model, exp1_history = train_blip(
    report_train_records,
    report_val_records,
    EXP1_DIR,
    MAX_REPORT_TOKENS,
    EXP1_MAX_STEPS,
)
display(exp1_history)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Gradient checkpointing enabled.


## 10. Experiment 2 — train the same BLIP base checkpoint on VQA supervision

This starts again from `Salesforce/blip-vqa-base`. It does not continue from Experiment 1.

In [ ]:
del exp1_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

exp2_model, exp2_history = train_blip(
    vqa_train_records,
    vqa_val_records,
    EXP2_DIR,
    MAX_VQA_ANSWER_TOKENS,
    EXP2_MAX_STEPS,
)
display(exp2_history)

## 11. Shared generation and text metrics

In [ ]:
@torch.inference_mode()
def generate_answers(model, records, max_new_tokens, batch_size=EVAL_BATCH_SIZE):
    device, amp_dtype = device_and_amp()
    model = model.to(device).eval()

    loader = DataLoader(
        BerdBlipDataset(records),
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=BlipInferenceCollator(blip_processor),
    )

    rows = []
    for batch in tqdm(loader, desc="Generating"):
        model_inputs = {
            k: v.to(device, non_blocking=True)
            for k, v in batch["model_inputs"].items()
        }
        with torch.autocast(
            device_type=device.type,
            dtype=amp_dtype if device.type == "cuda" else torch.float32,
            enabled=(device.type == "cuda"),
        ):
            generated = model.generate(
                **model_inputs,
                max_new_tokens=max_new_tokens,
                num_beams=3,
                early_stopping=True,
            )

        preds = blip_processor.batch_decode(generated, skip_special_tokens=True)
        for rec, pred in zip(batch["records"], preds):
            row = dict(rec)
            row["prediction"] = pred.strip()
            rows.append(row)
    return pd.DataFrame(rows)

def normalize_text(s):
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9\s,.-]", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def token_f1_single(pred, gold):
    p = normalize_text(pred).split()
    g = normalize_text(gold).split()
    if not p and not g:
        return 1.0
    if not p or not g:
        return 0.0
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def vqa_text_metrics(df):
    em = [
        float(normalize_text(p) == normalize_text(g))
        for p, g in zip(df.prediction, df.answer)
    ]
    f1 = [token_f1_single(p, g) for p, g in zip(df.prediction, df.answer)]
    return {"exact_match": float(np.mean(em)), "token_f1": float(np.mean(f1))}

## 12. Primary benchmark — BERD clinician-reviewed labels

Both models answer the exact same canonical question on the exact same test images. Their output is mapped with the same deterministic BERD-label parser.

In [ ]:
LABEL_ALIASES = {
    "neoplasm": ["neoplasm", "tumor", "tumour"],
    "narrow": ["narrow", "narrowing", "stenosis", "stenotic"],
    "blood": ["blood", "bleeding", "hemorrhage", "haemorrhage"],
    "clot": ["clot"],
    "sputum": ["sputum", "secretion", "secretions", "mucus", "mucous"],
    "rough": ["rough"],
    "normal": ["normal"],
    "infiltration changes": ["infiltration changes", "infiltration", "infiltrative"],
    "congested": ["congested", "congestion", "hyperemic", "hyperemia"],
    "edematous": ["edematous", "oedematous", "edema", "oedema"],
    "widened": ["widened", "widening", "dilated", "dilation"],
    "external pressure": ["external pressure", "extrinsic compression", "external compression"],
    "surgical stump": ["surgical stump", "stump"],
    "nodules": ["nodules", "nodule", "nodular"],
    "fistula": ["fistula"],
    "postoperative change": ["postoperative change", "post-operative change", "postsurgical change"],
    "granulation": ["granulation"],
    "necrotic": ["necrotic", "necrosis"],
    "mass": ["mass"],
    "pigmentation": ["pigmentation", "pigmented"],
    "tube": ["tube", "stent"],
    "ulcer": ["ulcer", "ulceration"],
}

def extract_berd_labels(text):
    t = normalize_text(text)
    found = set()
    for canonical, aliases in LABEL_ALIASES.items():
        if any(re.search(rf"(?<!\w){re.escape(alias)}(?!\w)", t) for alias in aliases):
            found.add(canonical)
    if len(found) > 1 and "normal" in found:
        found.remove("normal")
    return found

def gold_label_set(label_norm):
    return set(split_labels(label_norm))

def multilabel_metrics_from_predictions(df):
    gold_sets = [gold_label_set(x) for x in df["gold_labels"]]
    pred_sets = [extract_berd_labels(x) for x in df["prediction"]]
    universe = sorted(set().union(*gold_sets, *pred_sets) | set(OFFICIAL_BERD_LABELS))

    mlb = MultiLabelBinarizer(classes=universe)
    mlb.fit([universe])
    y_true = mlb.transform(gold_sets)
    y_pred = mlb.transform(pred_sets)

    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="micro", zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    return {
        "label_set_exact_match": float(np.mean([g == p for g, p in zip(gold_sets, pred_sets)])),
        "label_micro_precision": float(p_micro),
        "label_micro_recall": float(r_micro),
        "label_micro_f1": float(f1_micro),
        "label_macro_precision": float(p_macro),
        "label_macro_recall": float(r_macro),
        "label_macro_f1": float(f1_macro),
    }

canonical_test_records = [
    {
        "image_id": str(r.image_id),
        "patient_id": str(r.patient_id),
        "image_file": str(r.image_file),
        "question": CANONICAL_QUESTION,
        "answer": str(r.label_norm),
        "gold_labels": str(r.label_norm),
        "source_caption": str(r.caption),
        "task": "gold_label_vqa",
    }
    for r in test_df.itertuples(index=False)
]

In [ ]:
device, _ = device_and_amp()

exp1_model = BlipForQuestionAnswering.from_pretrained(EXP1_DIR).to(device)
exp1_gold_pred = generate_answers(exp1_model, canonical_test_records, MAX_REPORT_TOKENS)
exp1_gold_metrics = multilabel_metrics_from_predictions(exp1_gold_pred)

exp1_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

exp2_model = BlipForQuestionAnswering.from_pretrained(EXP2_DIR).to(device)
exp2_gold_pred = generate_answers(exp2_model, canonical_test_records, MAX_VQA_ANSWER_TOKENS)
exp2_gold_metrics = multilabel_metrics_from_predictions(exp2_gold_pred)

display(pd.DataFrame([
    {"experiment": "Exp1_report", **exp1_gold_metrics},
    {"experiment": "Exp2_vqa", **exp2_gold_metrics},
]))

### Side-by-side gold-label predictions

In [ ]:
side = (
    exp1_gold_pred[["image_id", "gold_labels", "source_caption", "prediction"]]
    .rename(columns={"prediction": "exp1_prediction"})
    .merge(
        exp2_gold_pred[["image_id", "prediction"]].rename(columns={"prediction": "exp2_prediction"}),
        on="image_id",
    )
)
side["exp1_parsed_labels"] = side["exp1_prediction"].map(lambda x: ", ".join(sorted(extract_berd_labels(x))))
side["exp2_parsed_labels"] = side["exp2_prediction"].map(lambda x: ", ".join(sorted(extract_berd_labels(x))))
display(side.head(40))

## 13. Held-out generated VQA benchmark

In [ ]:
exp2_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

exp1_model = BlipForQuestionAnswering.from_pretrained(EXP1_DIR).to(device)
exp1_vqa_pred = generate_answers(exp1_model, vqa_test_records, MAX_VQA_ANSWER_TOKENS)
exp1_vqa_metrics = vqa_text_metrics(exp1_vqa_pred)

exp1_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

exp2_model = BlipForQuestionAnswering.from_pretrained(EXP2_DIR).to(device)
exp2_vqa_pred = generate_answers(exp2_model, vqa_test_records, MAX_VQA_ANSWER_TOKENS)
exp2_vqa_metrics = vqa_text_metrics(exp2_vqa_pred)

display(pd.DataFrame([
    {"experiment": "Exp1_report", **exp1_vqa_metrics},
    {"experiment": "Exp2_vqa", **exp2_vqa_metrics},
]))

## 14. Original BERD caption benchmark

In [ ]:
def report_metrics(pred_df):
    refs = [str(x) for x in pred_df["answer"]]
    preds = [str(x) for x in pred_df["prediction"]]

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score / 100.0

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = np.mean([
        scorer.score(ref, pred)["rougeL"].fmeasure
        for ref, pred in zip(refs, preds)
    ])

    try:
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        meteor = np.mean([
            meteor_score([ref.split()], pred.split())
            for ref, pred in zip(refs, preds)
        ])
    except Exception as e:
        print("METEOR unavailable:", e)
        meteor = np.nan

    return {"BLEU": float(bleu), "ROUGE_L": float(rouge_l), "METEOR": float(meteor)}

exp2_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

exp1_model = BlipForQuestionAnswering.from_pretrained(EXP1_DIR).to(device)
exp1_report_pred = generate_answers(exp1_model, report_test_records, MAX_REPORT_TOKENS)
exp1_report_metrics = report_metrics(exp1_report_pred)

exp1_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()

exp2_model = BlipForQuestionAnswering.from_pretrained(EXP2_DIR).to(device)
exp2_report_pred = generate_answers(exp2_model, report_test_records, MAX_REPORT_TOKENS)
exp2_report_metrics = report_metrics(exp2_report_pred)

display(pd.DataFrame([
    {"experiment": "Exp1_report", **exp1_report_metrics},
    {"experiment": "Exp2_vqa", **exp2_report_metrics},
]))

## 15. Paired bootstrap confidence interval for the primary exact-match delta

This estimates the 95% CI of `Exp2 - Exp1` on the same test images.

In [ ]:
def correctness_vector(pred_df):
    gold_sets = [gold_label_set(x) for x in pred_df["gold_labels"]]
    pred_sets = [extract_berd_labels(x) for x in pred_df["prediction"]]
    return np.array([g == p for g, p in zip(gold_sets, pred_sets)], dtype=float)

def paired_bootstrap_delta(a, b, n_boot=10000, seed=SEED):
    assert len(a) == len(b)
    rng = np.random.default_rng(seed)
    n = len(a)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        deltas[i] = b[idx].mean() - a[idx].mean()
    return {
        "delta": float(b.mean() - a.mean()),
        "ci95_low": float(np.percentile(deltas, 2.5)),
        "ci95_high": float(np.percentile(deltas, 97.5)),
        "prob_delta_gt_0": float(np.mean(deltas > 0)),
    }

bootstrap_result = paired_bootstrap_delta(
    correctness_vector(exp1_gold_pred),
    correctness_vector(exp2_gold_pred),
)
bootstrap_result

## 16. Final comparison table and saved outputs

In [ ]:
rows = []

def add_metric(benchmark, metric, exp1, exp2):
    rows.append({
        "benchmark": benchmark,
        "metric": metric,
        "exp1_report": float(exp1),
        "exp2_vqa": float(exp2),
        "delta_exp2_minus_exp1": float(exp2 - exp1),
    })

add_metric("BERD gold labels", "label_set_exact_match",
           exp1_gold_metrics["label_set_exact_match"], exp2_gold_metrics["label_set_exact_match"])
add_metric("BERD gold labels", "label_micro_f1",
           exp1_gold_metrics["label_micro_f1"], exp2_gold_metrics["label_micro_f1"])
add_metric("BERD gold labels", "label_macro_f1",
           exp1_gold_metrics["label_macro_f1"], exp2_gold_metrics["label_macro_f1"])
add_metric("Held-out generated VQA", "exact_match",
           exp1_vqa_metrics["exact_match"], exp2_vqa_metrics["exact_match"])
add_metric("Held-out generated VQA", "token_f1",
           exp1_vqa_metrics["token_f1"], exp2_vqa_metrics["token_f1"])
add_metric("Original report", "BLEU",
           exp1_report_metrics["BLEU"], exp2_report_metrics["BLEU"])
add_metric("Original report", "ROUGE_L",
           exp1_report_metrics["ROUGE_L"], exp2_report_metrics["ROUGE_L"])
add_metric("Original report", "METEOR",
           exp1_report_metrics["METEOR"], exp2_report_metrics["METEOR"])

comparison_df = pd.DataFrame(rows)
display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "comparison.csv", index=False)
with open(RESULTS_DIR / "bootstrap_primary_exact_match.json", "w") as f:
    json.dump(bootstrap_result, f, indent=2)

exp1_gold_pred.to_csv(RESULTS_DIR / "exp1_gold_predictions.csv", index=False)
exp2_gold_pred.to_csv(RESULTS_DIR / "exp2_gold_predictions.csv", index=False)
exp1_vqa_pred.to_csv(RESULTS_DIR / "exp1_vqa_predictions.csv", index=False)
exp2_vqa_pred.to_csv(RESULTS_DIR / "exp2_vqa_predictions.csv", index=False)
exp1_report_pred.to_csv(RESULTS_DIR / "exp1_report_predictions.csv", index=False)
exp2_report_pred.to_csv(RESULTS_DIR / "exp2_report_predictions.csv", index=False)

print("Saved results to:", RESULTS_DIR)
print(json.dumps(bootstrap_result, indent=2))

## 17. Plot the comparison

In [ ]:
import matplotlib.pyplot as plt

plot_df = comparison_df.copy()
x = np.arange(len(plot_df))
width = 0.38

plt.figure(figsize=(14, 6))
plt.bar(x - width/2, plot_df["exp1_report"], width, label="Exp1 report")
plt.bar(x + width/2, plot_df["exp2_vqa"], width, label="Exp2 VQA")
plt.xticks(
    x,
    [f"{b}\n{m}" for b, m in zip(plot_df["benchmark"], plot_df["metric"])],
    rotation=35,
    ha="right",
)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("BLIP on BERD: report supervision vs VQA supervision")
plt.legend()
plt.tight_layout()
plt.show()

## 18. How to interpret the experiment

The strongest evidence that VQA reformulation helps is:

- higher **BERD gold-label micro-F1 and exact match** for Experiment 2,
- a paired-bootstrap confidence interval for `Exp2 - Exp1` that is mostly or entirely above 0,
- the gain repeats across several random seeds,
- and manual audit confirms that the generated QA supervision is visually grounded.

Do **not** conclude that the method works only because Experiment 2 performs better on the LLM-generated QA benchmark; that benchmark partly reflects the formatting style created by the LLM.

### Recommended ablations

Run at least these:

1. **Report baseline** — this notebook's Experiment 1.
2. **Canonical-only VQA** — only the BERD gold-label QA pair, no LLM extra pairs.
3. **Canonical + LLM VQA** — this notebook's Experiment 2.
4. Repeat with 3+ seeds.
5. Optionally compare equal-steps and equal-epochs training.

The canonical-only ablation is particularly important: it tells you whether improvement comes simply from shortening the answer to clinician labels or from the additional LLM-generated question diversity.

## 19. BERD-specific limitations to include in your write-up

- BERD was collected at a single hospital, so external generalization is not established.
- The release contains patient IDs because multiple images can belong to one patient; preserve patient-disjoint splitting.
- Some numerical details were removed because they are not reliably inferable from a single image.
- The English image captions were translated from Chinese during dataset construction.
- VQA reformulation may improve short-answer recognition while reducing report richness, which is why both recognition and report-generation metrics are reported here.